# nb42 — Subgroup Analysis: DiD · Lift · CD5

Three metrics × three subgroup comparisons:

| Comparison | DiD (citations) | Lift | CD5 |
|---|---|---|---|
| Awardees vs Non-awardees | ✓ | ✓ | ✓ |
| Juniors vs Seniors (awardees only) | ✓ | ✓ | ✓ |
| Junior award paper vs Senior award paper | ✓ | ✓ | ✓ |

**Seniority**: `career_age_at_award > 5 years` → Senior, else Junior  
**Career age** = `award_year − first_pub_year` from nb37 OpenAlex profiles  
**Event study**: ±5 window, reference t=−1, HC3 SEs  
**Lift** = `post_avg_citations / pre_avg_citations` (windows: pre = [−5,−1], post = [+1,+5])


## Section 0 — Load Data & Build Master Panel

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

ROOT   = Path('..')
CD_DIR = ROOT / 'data' / 'cd_trajectory'
FIG_DIR = ROOT / 'data' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── raw panel & metrics ──────────────────────────────────────────────────────
panel   = pd.read_csv(CD_DIR / 'author_papers_panel.csv')
cd5     = pd.read_csv(CD_DIR / 'paper_cd5_scores.csv')
cit_cnt = pd.read_csv(CD_DIR / 'papers_citation_counts.csv')

print('panel  :', panel.shape,  panel.columns.tolist())
print('cd5    :', cd5.shape,    cd5.columns.tolist())
print('cit_cnt:', cit_cnt.shape, cit_cnt.columns.tolist())

In [ ]:
# ── seniority: load real first_pub_year from nb37 OpenAlex profiles ──────────
# (deriving from panel paper_year is wrong — panel only covers ±5 window)
profiles = pd.read_csv(ROOT / 'data' / 'raw' / 'icwsm_jcdl_author_profiles_clean.csv')

profiles_clean = (
    profiles[profiles['career_age_valid'] == True]
    [['author_id', 'first_pub_year']]
    .drop_duplicates('author_id')
)

award_info = (
    panel[panel['group'] == 'award']
         .drop_duplicates('author_id')[['author_id', 'award_year']]
)

seniority = award_info.merge(profiles_clean, on='author_id', how='inner')
seniority['career_age'] = seniority['award_year'] - seniority['first_pub_year']
seniority['is_senior']  = (seniority['career_age'] > 5).astype(int)  # 1=Senior, 0=Junior

print(seniority[['career_age','is_senior']].describe())
print('\nJuniors:', (seniority['is_senior']==0).sum(),
      '| Seniors:', (seniority['is_senior']==1).sum())

In [ ]:
# ── merge citation counts onto panel, then aggregate to author-year ───────────
# panel has no 'citations' column — cited_by_count lives in papers_citation_counts.csv
panel = panel.merge(
    cit_cnt[['paper_id', 'cited_by_count']],
    on='paper_id',
    how='left'
).rename(columns={'cited_by_count': 'citations'})

cit_ay = (
    panel.groupby(['author_id', 'group', 'award_year', 'relative_year'])['citations']
         .sum()
         .reset_index()
)
print('Citation author-year panel:', cit_ay.shape)
print(cit_ay.head(3))

# ── Also check author_year_cd5.csv — it may already be aggregated ──────────
cd5_ready = pd.read_csv(CD_DIR / 'author_year_cd5.csv')
print('\nauthor_year_cd5 cols:', cd5_ready.columns.tolist())
print(cd5_ready.head(3))

In [ ]:
# ── build author-year CD5 panel ───────────────────────────────────────────────
# merge cd5 scores onto paper panel to get (author_id, relative_year, cd5)
if 'paper_id' not in panel.columns:
    # panel might use 'work_id'
    panel_key = [c for c in panel.columns if 'id' in c.lower() and 'author' not in c.lower()][0]
    cd5_key   = [c for c in cd5.columns   if 'id' in c.lower() and 'author' not in c.lower()][0]
else:
    panel_key = 'paper_id'
    cd5_key   = 'paper_id'

panel_cd5 = panel.merge(cd5.rename(columns={cd5_key: panel_key}), on=panel_key, how='inner')

cd5_ay = (
    panel_cd5
        .groupby(['author_id','group','award_year','relative_year'])['cd5']
        .mean()
        .reset_index()
)

print('CD5 author-year panel:', cd5_ay.shape)
print(cd5_ay.head(3))

In [ ]:
# ── compute lift per author ───────────────────────────────────────────────────
PRE  = range(-5, 0)    # t=-5...-1
POST = range(1, 6)     # t=+1...+5

def window_mean(df, aid, window, col='citations'):
    sub = df[(df['author_id']==aid) & (df['relative_year'].isin(window))]
    return sub[col].mean() if len(sub) else np.nan

authors_cit = cit_ay[cit_ay['relative_year'].isin(list(PRE)+list(POST))]
auth_ids = authors_cit['author_id'].unique()

lift_rows = []
for aid in auth_ids:
    pre  = window_mean(authors_cit, aid, PRE)
    post = window_mean(authors_cit, aid, POST)
    if pre and pre > 0 and post and not np.isnan(post):
        g = authors_cit.loc[authors_cit['author_id']==aid, 'group'].iloc[0]
        lift_rows.append({'author_id': aid, 'group': g,
                          'pre_avg': pre, 'post_avg': post,
                          'lift': post/pre})

lift_df = pd.DataFrame(lift_rows)
lift_df = lift_df.merge(seniority[['author_id','is_senior']], on='author_id', how='left')

print('Lift records:', len(lift_df))
print(lift_df.groupby('group')['lift'].median())

## Section 1 — Awardees vs Non-awardees

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# SHARED HELPERS
# ────────────────────────────────────────────────────────────────────────────

def run_event_study(df, outcome_col, group_col, window=5, ref=-1):
    """
    TWFE event study.
    df must have: author_id, relative_year, award_year, <outcome_col>, <group_col> (0/1)
    Returns tidy DataFrame of (rel_year, coef, ci_lo, ci_hi).
    """
    sub = df[df['relative_year'].between(-window, window)].copy()
    sub = sub[sub['relative_year'] != ref]          # drop reference year
    sub['post'] = (sub['relative_year'] > 0).astype(int)

    # author & year FE via dummy encoding (small data – no absorb)
    # interaction: treated * C(relative_year)
    import statsmodels.formula.api as smf
    formula = f'{outcome_col} ~ C(relative_year)*C({group_col}) + C(author_id) + C(award_year)'
    try:
        model = smf.ols(formula, data=sub).fit(cov_type='HC3')
    except Exception as e:
        print('Event study error:', e)
        return None, None

    # extract interaction coefficients
    rows = []
    for t in range(-window, window+1):
        if t == ref:
            rows.append({'rel_year': t, 'coef': 0.0, 'ci_lo': 0.0, 'ci_hi': 0.0})
            continue
        key = f'C(relative_year)[T.{t}]:C({group_col})[T.1]'
        if key in model.params:
            rows.append({
                'rel_year': t,
                'coef'  : model.params[key],
                'ci_lo' : model.conf_int().loc[key, 0],
                'ci_hi' : model.conf_int().loc[key, 1],
            })
        else:
            rows.append({'rel_year': t, 'coef': np.nan, 'ci_lo': np.nan, 'ci_hi': np.nan})
    return pd.DataFrame(rows), model


def simple_did(df, outcome_col, group_col):
    """Simple 2×2 DiD: post × treated interaction."""
    sub = df[df['relative_year'].between(-5, 5)].copy()
    sub['post']    = (sub['relative_year'] > 0).astype(int)
    formula = f'{outcome_col} ~ post * C({group_col}) + C(author_id) + C(award_year)'
    model = smf.ols(formula, data=sub).fit(cov_type='HC3')
    key = f'post:C({group_col})[T.1]'
    coef = model.params.get(key, np.nan)
    pval = model.pvalues.get(key, np.nan)
    ci   = model.conf_int().loc[key] if key in model.conf_int().index else (np.nan, np.nan)
    return coef, pval, ci, model


def pretrend_test(df, outcome_col, group_col):
    """Pre-trend test: slope of outcome ~ rel_year*treated in pre-period only."""
    sub = df[(df['relative_year'] < 0) & (df['relative_year'] >= -5)].copy()
    formula = f'{outcome_col} ~ relative_year * C({group_col}) + C(author_id)'
    try:
        m = smf.ols(formula, data=sub).fit(cov_type='HC3')
        key = f'relative_year:C({group_col})[T.1]'
        p = m.pvalues.get(key, np.nan)
        return p
    except:
        return np.nan


def plot_event_study(es_df, title, ylabel, save_path):
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.errorbar(
        es_df['rel_year'], es_df['coef'],
        yerr=[es_df['coef']-es_df['ci_lo'], es_df['ci_hi']-es_df['coef']],
        fmt='o-', color='steelblue', ecolor='lightsteelblue',
        elinewidth=2, capsize=4, linewidth=1.5, markersize=5
    )
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.axvline(0, color='grey',  linewidth=0.8, linestyle=':')
    ax.set_xlabel('Years relative to award', fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f'Saved → {save_path}')


def lift_comparison(lift_df, group_col, group_label_map, title_suffix, save_path):
    """Box/strip plot + Mann-Whitney U for lift."""
    from scipy.stats import mannwhitneyu
    grp_vals = sorted(lift_df[group_col].dropna().unique())
    data = [lift_df.loc[lift_df[group_col]==v, 'lift'].dropna().values for v in grp_vals]
    labels = [group_label_map.get(v, str(v)) for v in grp_vals]

    stat, p = mannwhitneyu(data[0], data[1], alternative='two-sided') if len(data)==2 else (np.nan, np.nan)
    print(f'\nLift | {title_suffix}')
    for v, d in zip(labels, data):
        print(f'  {v}: n={len(d):4d}  median={np.median(d):.3f}  mean={np.mean(d):.3f}')
    print(f'  Mann-Whitney U p={p:.4f}')

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.boxplot(data, labels=labels, patch_artist=True,
               boxprops=dict(facecolor='lightsteelblue'),
               medianprops=dict(color='navy', linewidth=2))
    ax.set_ylabel('Citation Lift (post/pre)', fontsize=11)
    ax.set_title(f'Lift — {title_suffix}', fontsize=12, fontweight='bold')
    ax.text(0.98, 0.96, f'p={p:.4f}', transform=ax.transAxes,
            ha='right', va='top', fontsize=10, color='darkred')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f'Saved → {save_path}')
    return p


print('Helpers defined.')

In [ ]:
# ── 1.A  DiD: Awardees vs Non-awardees ──────────────────────────────────────
cit_ay['is_awardee'] = (cit_ay['group'] == 'award').astype(int)
cit_win = cit_ay[cit_ay['relative_year'].between(-5, 5)].copy()

es_df, model_1a = run_event_study(cit_win, 'citations', 'is_awardee')
plot_event_study(
    es_df,
    title='Event Study: Citations — Awardees vs Non-Awardees',
    ylabel='Δ Citations (vs non-awardee, ref t=−1)',
    save_path=FIG_DIR / '42_es_citations_award_vs_nonaward.png'
)

coef, pval, ci, _ = simple_did(cit_win, 'citations', 'is_awardee')
p_pre             = pretrend_test(cit_win, 'citations', 'is_awardee')
print(f'\n2×2 DiD coef={coef:.3f}  p={pval:.4f}  CI=[{ci[0]:.3f}, {ci[1]:.3f}]')
print(f'Pre-trend p={p_pre:.4f}  → {"✓ Parallel" if p_pre>0.1 else "⚠ Non-parallel"}')

In [ ]:
# ── 1.B  Lift: Awardees vs Non-awardees ─────────────────────────────────────
lift_df['is_awardee'] = (lift_df['group'] == 'award').astype(int)
p_lift_1b = lift_comparison(
    lift_df, 'is_awardee',
    {0: 'Non-awardee', 1: 'Awardee'},
    'Awardees vs Non-Awardees',
    FIG_DIR / '42_lift_award_vs_nonaward.png'
)

In [ ]:
# ── 1.C  CD5: Awardees vs Non-awardees ──────────────────────────────────────
cd5_ay['is_awardee'] = (cd5_ay['group'] == 'award').astype(int)
cd5_win = cd5_ay[cd5_ay['relative_year'].between(-5, 5)].copy()

es_cd5_1c, model_1c = run_event_study(cd5_win, 'cd5', 'is_awardee')
plot_event_study(
    es_cd5_1c,
    title='Event Study: CD5 — Awardees vs Non-Awardees',
    ylabel='Δ CD5 (vs non-awardee, ref t=−1)',
    save_path=FIG_DIR / '42_es_cd5_award_vs_nonaward.png'
)

coef_c, pval_c, ci_c, _ = simple_did(cd5_win, 'cd5', 'is_awardee')
p_pre_c                  = pretrend_test(cd5_win, 'cd5', 'is_awardee')
print(f'\n2×2 DiD coef={coef_c:.4f}  p={pval_c:.4f}  CI=[{ci_c[0]:.4f}, {ci_c[1]:.4f}]')
print(f'Pre-trend p={p_pre_c:.4f}  → {"✓ Parallel" if p_pre_c>0.1 else "⚠ Non-parallel"}')

## Section 2 — Juniors vs Seniors (awardees only)

In [ ]:
# merge seniority onto citation & CD5 panels (awardees only)
cit_awd  = cit_ay[cit_ay['group']=='award'].merge(
                seniority[['author_id','is_senior']], on='author_id', how='inner')
cd5_awd  = cd5_ay[cd5_ay['group']=='award'].merge(
                seniority[['author_id','is_senior']], on='author_id', how='inner')

cit_awd_win = cit_awd[cit_awd['relative_year'].between(-5,5)].copy()
cd5_awd_win = cd5_awd[cd5_awd['relative_year'].between(-5,5)].copy()

print('Awardee citation panel:', cit_awd_win.shape)
print('Awardee CD5 panel     :', cd5_awd_win.shape)

In [ ]:
# ── 2.A  DiD: Juniors vs Seniors ────────────────────────────────────────────
es_2a, model_2a = run_event_study(cit_awd_win, 'citations', 'is_senior')
plot_event_study(
    es_2a,
    title='Event Study: Citations — Seniors vs Juniors (awardees)',
    ylabel='Δ Citations (Senior − Junior, ref t=−1)',
    save_path=FIG_DIR / '42_es_citations_senior_vs_junior.png'
)

coef_2a, pval_2a, ci_2a, _ = simple_did(cit_awd_win, 'citations', 'is_senior')
p_pre_2a                    = pretrend_test(cit_awd_win, 'citations', 'is_senior')
print(f'\n2×2 DiD coef={coef_2a:.3f}  p={pval_2a:.4f}  CI=[{ci_2a[0]:.3f}, {ci_2a[1]:.3f}]')
print(f'Pre-trend p={p_pre_2a:.4f}  → {"✓ Parallel" if p_pre_2a>0.1 else "⚠ Non-parallel"}')

In [ ]:
# ── 2.B  Lift: Juniors vs Seniors ────────────────────────────────────────────
lift_awd = lift_df[lift_df['group']=='award'].copy()
p_lift_2b = lift_comparison(
    lift_awd, 'is_senior',
    {0: 'Junior', 1: 'Senior'},
    'Seniors vs Juniors (awardees)',
    FIG_DIR / '42_lift_senior_vs_junior.png'
)

In [ ]:
# ── 2.C  CD5: Juniors vs Seniors ─────────────────────────────────────────────
es_2c, model_2c = run_event_study(cd5_awd_win, 'cd5', 'is_senior')
plot_event_study(
    es_2c,
    title='Event Study: CD5 — Seniors vs Juniors (awardees)',
    ylabel='Δ CD5 (Senior − Junior, ref t=−1)',
    save_path=FIG_DIR / '42_es_cd5_senior_vs_junior.png'
)

coef_2c, pval_2c, ci_2c, _ = simple_did(cd5_awd_win, 'cd5', 'is_senior')
p_pre_2c                    = pretrend_test(cd5_awd_win, 'cd5', 'is_senior')
print(f'\n2×2 DiD coef={coef_2c:.4f}  p={pval_2c:.4f}  CI=[{ci_2c[0]:.4f}, {ci_2c[1]:.4f}]')
print(f'Pre-trend p={p_pre_2c:.4f}  → {"✓ Parallel" if p_pre_2c>0.1 else "⚠ Non-parallel"}')

## Section 3 — Junior Award Paper vs Senior Award Paper

> **Definition**: split award papers by the *seniority of the awardee author*.
> A 'Junior award paper' = an award paper whose author had career age ≤ 5 at award time.
> We compare trajectories of the two groups against their matched controls.


In [ ]:
# For each control author, we need to know which awardee they are matched to,
# so we can inherit seniority.  Fall back: tag controls by the group's median career age.
#
# Simpler workable approach:
#   - For the award group, seniority is known directly.
#   - We run two separate event studies:
#       (a) Junior awardees  vs their controls
#       (b) Senior awardees  vs their controls
#   using is_awardee as treatment variable, restricted to the seniority subset.

# get junior/senior awardee author_ids
junior_awd_ids = seniority.loc[seniority['is_senior']==0, 'author_id']
senior_awd_ids = seniority.loc[seniority['is_senior']==1, 'author_id']

# controls: split equally or just use all controls for both comparisons
# (standard approach: use all control authors as baseline for each sub-comparison)
ctrl_ids = cit_ay.loc[cit_ay['group']=='control', 'author_id'].unique()

def make_subset(panel_df, target_author_ids, ctrl_ids, outcome_col):
    awd_sub  = panel_df[panel_df['author_id'].isin(target_author_ids)].copy()
    ctrl_sub = panel_df[panel_df['author_id'].isin(ctrl_ids)].copy()
    combined = pd.concat([awd_sub, ctrl_sub], ignore_index=True)
    combined['is_awardee'] = (combined['group']=='award').astype(int)
    return combined[combined['relative_year'].between(-5,5)].copy()

cit_junior = make_subset(cit_ay, junior_awd_ids, ctrl_ids, 'citations')
cit_senior = make_subset(cit_ay, senior_awd_ids, ctrl_ids, 'citations')
cd5_junior = make_subset(cd5_ay, junior_awd_ids, ctrl_ids, 'cd5')
cd5_senior = make_subset(cd5_ay, senior_awd_ids, ctrl_ids, 'cd5')

print('Junior award paper citation panel:', cit_junior.shape)
print('Senior award paper citation panel:', cit_senior.shape)

In [ ]:
# ── 3.A  DiD: Junior Award Paper ─────────────────────────────────────────────
es_3a_jnr, _ = run_event_study(cit_junior, 'citations', 'is_awardee')
es_3a_snr, _ = run_event_study(cit_senior, 'citations', 'is_awardee')

# overlay plot
fig, ax = plt.subplots(figsize=(10, 5))
for es, label, col in [
    (es_3a_jnr, 'Junior award paper', 'steelblue'),
    (es_3a_snr, 'Senior award paper', 'coral'),
]:
    if es is not None:
        ax.errorbar(
            es['rel_year'], es['coef'],
            yerr=[es['coef']-es['ci_lo'], es['ci_hi']-es['coef']],
            fmt='o-', label=label, color=col, ecolor=col,
            alpha=0.7, elinewidth=1.5, capsize=3, linewidth=1.5, markersize=5
        )
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.axvline(0, color='grey',  linewidth=0.8, linestyle=':')
ax.set_xlabel('Years relative to award', fontsize=12)
ax.set_ylabel('Δ Citations (vs controls, ref t=−1)', fontsize=12)
ax.set_title('Event Study: Citations — Junior vs Senior Award Paper', fontsize=13, fontweight='bold')
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(FIG_DIR / '42_es_citations_jnr_vs_snr_awardpaper.png', dpi=150)
plt.show()

for lbl, df_sub in [('Junior award paper', cit_junior), ('Senior award paper', cit_senior)]:
    coef, pval, ci, _ = simple_did(df_sub, 'citations', 'is_awardee')
    p_pre             = pretrend_test(df_sub, 'citations', 'is_awardee')
    print(f'\n{lbl}: 2×2 DiD coef={coef:.3f}  p={pval:.4f}  CI=[{ci[0]:.3f}, {ci[1]:.3f}]  pre-trend p={p_pre:.4f}')

In [ ]:
# ── 3.B  Lift: Junior vs Senior Award Paper ───────────────────────────────────
lift_awd2 = lift_df[lift_df['group']=='award'].copy()
# is_senior already present from Section 0
p_lift_3b = lift_comparison(
    lift_awd2, 'is_senior',
    {0: 'Junior award paper', 1: 'Senior award paper'},
    'Junior vs Senior Award Paper',
    FIG_DIR / '42_lift_jnr_vs_snr_awardpaper.png'
)

In [ ]:
# ── 3.C  CD5: Junior vs Senior Award Paper ────────────────────────────────────
es_3c_jnr, _ = run_event_study(cd5_junior, 'cd5', 'is_awardee')
es_3c_snr, _ = run_event_study(cd5_senior, 'cd5', 'is_awardee')

fig, ax = plt.subplots(figsize=(10, 5))
for es, label, col in [
    (es_3c_jnr, 'Junior award paper', 'steelblue'),
    (es_3c_snr, 'Senior award paper', 'coral'),
]:
    if es is not None:
        ax.errorbar(
            es['rel_year'], es['coef'],
            yerr=[es['coef']-es['ci_lo'], es['ci_hi']-es['coef']],
            fmt='o-', label=label, color=col, ecolor=col,
            alpha=0.7, elinewidth=1.5, capsize=3, linewidth=1.5, markersize=5
        )
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.axvline(0, color='grey',  linewidth=0.8, linestyle=':')
ax.set_xlabel('Years relative to award', fontsize=12)
ax.set_ylabel('Δ CD5 (vs controls, ref t=−1)', fontsize=12)
ax.set_title('Event Study: CD5 — Junior vs Senior Award Paper', fontsize=13, fontweight='bold')
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(FIG_DIR / '42_es_cd5_jnr_vs_snr_awardpaper.png', dpi=150)
plt.show()

for lbl, df_sub in [('Junior award paper', cd5_junior), ('Senior award paper', cd5_senior)]:
    coef, pval, ci, _ = simple_did(df_sub, 'cd5', 'is_awardee')
    p_pre             = pretrend_test(df_sub, 'cd5', 'is_awardee')
    print(f'\n{lbl}: 2×2 DiD coef={coef:.4f}  p={pval:.4f}  CI=[{ci[0]:.4f}, {ci[1]:.4f}]  pre-trend p={p_pre:.4f}')

## Section 4 — Summary Table of All 9 DiD Coefficients

In [ ]:
from scipy.stats import mannwhitneyu

# collect all results
results = []

# 1A – DiD citations award vs non
c,p,ci,_ = simple_did(cit_win, 'citations', 'is_awardee')
results.append({'Comparison':'Awardee vs Non-awardee','Metric':'Citations (DiD)',
                'Coef/Stat':round(c,3),'p-value':round(p,4),'CI':f'[{ci[0]:.2f}, {ci[1]:.2f}]'})

# 1B – Lift award vs non
g0 = lift_df.loc[lift_df['is_awardee']==0,'lift'].dropna()
g1 = lift_df.loc[lift_df['is_awardee']==1,'lift'].dropna()
stat,p_l = mannwhitneyu(g0, g1, alternative='two-sided')
results.append({'Comparison':'Awardee vs Non-awardee','Metric':'Lift (MWU stat)',
                'Coef/Stat':round(stat,1),'p-value':round(p_l,4),'CI':'—'})

# 1C – CD5 award vs non
c,p,ci,_ = simple_did(cd5_win, 'cd5', 'is_awardee')
results.append({'Comparison':'Awardee vs Non-awardee','Metric':'CD5 (DiD)',
                'Coef/Stat':round(c,4),'p-value':round(p,4),'CI':f'[{ci[0]:.4f}, {ci[1]:.4f}]'})

# 2A – DiD citations senior vs junior
c,p,ci,_ = simple_did(cit_awd_win, 'citations', 'is_senior')
results.append({'Comparison':'Senior vs Junior (awardees)','Metric':'Citations (DiD)',
                'Coef/Stat':round(c,3),'p-value':round(p,4),'CI':f'[{ci[0]:.2f}, {ci[1]:.2f}]'})

# 2B – Lift senior vs junior
g0 = lift_awd.loc[lift_awd['is_senior']==0,'lift'].dropna()
g1 = lift_awd.loc[lift_awd['is_senior']==1,'lift'].dropna()
stat,p_l = mannwhitneyu(g0, g1, alternative='two-sided')
results.append({'Comparison':'Senior vs Junior (awardees)','Metric':'Lift (MWU stat)',
                'Coef/Stat':round(stat,1),'p-value':round(p_l,4),'CI':'—'})

# 2C – CD5 senior vs junior
c,p,ci,_ = simple_did(cd5_awd_win, 'cd5', 'is_senior')
results.append({'Comparison':'Senior vs Junior (awardees)','Metric':'CD5 (DiD)',
                'Coef/Stat':round(c,4),'p-value':round(p,4),'CI':f'[{ci[0]:.4f}, {ci[1]:.4f}]'})

# 3A – DiD citations junior award paper
c,p,ci,_ = simple_did(cit_junior, 'citations', 'is_awardee')
results.append({'Comparison':'Junior award paper vs controls','Metric':'Citations (DiD)',
                'Coef/Stat':round(c,3),'p-value':round(p,4),'CI':f'[{ci[0]:.2f}, {ci[1]:.2f}]'})
c,p,ci,_ = simple_did(cit_senior, 'citations', 'is_awardee')
results.append({'Comparison':'Senior award paper vs controls','Metric':'Citations (DiD)',
                'Coef/Stat':round(c,3),'p-value':round(p,4),'CI':f'[{ci[0]:.2f}, {ci[1]:.2f}]'})

# 3B – Lift
g0 = lift_awd2.loc[lift_awd2['is_senior']==0,'lift'].dropna()
g1 = lift_awd2.loc[lift_awd2['is_senior']==1,'lift'].dropna()
stat,p_l = mannwhitneyu(g0, g1, alternative='two-sided')
results.append({'Comparison':'Jnr/Snr award paper (vs controls)','Metric':'Lift (MWU stat)',
                'Coef/Stat':round(stat,1),'p-value':round(p_l,4),'CI':'—'})

# 3C – CD5 junior award paper
c,p,ci,_ = simple_did(cd5_junior, 'cd5', 'is_awardee')
results.append({'Comparison':'Junior award paper vs controls','Metric':'CD5 (DiD)',
                'Coef/Stat':round(c,4),'p-value':round(p,4),'CI':f'[{ci[0]:.4f}, {ci[1]:.4f}]'})
c,p,ci,_ = simple_did(cd5_senior, 'cd5', 'is_awardee')
results.append({'Comparison':'Senior award paper vs controls','Metric':'CD5 (DiD)',
                'Coef/Stat':round(c,4),'p-value':round(p,4),'CI':f'[{ci[0]:.4f}, {ci[1]:.4f}]'})

summary_df = pd.DataFrame(results)
print(summary_df.to_string(index=False))

out_path = ROOT / 'data' / 'cd_trajectory' / '42_subgroup_summary.csv'
summary_df.to_csv(out_path, index=False)
print(f'\n✓ Summary saved → {out_path}')